In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kanchandalal123/mcq-deberta-optimization-model/config.json
/kaggle/input/datasets/kanchandalal123/mcq-deberta-optimization-model/training_args.bin
/kaggle/input/datasets/kanchandalal123/mcq-deberta-optimization-model/tokenizer.json
/kaggle/input/datasets/kanchandalal123/mcq-deberta-optimization-model/tokenizer_config.json
/kaggle/input/datasets/kanchandalal123/mcq-deberta-optimization-model/model.safetensors
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver - DeBERTa Inference

This notebook loads the fine-tuned DeBERTa model from a Kaggle Dataset and generates predictions for the competition test set.

Workflow

- Load trained model
- Prepare ranking dataset
- Tokenize test data
- Predict option scores
- Rank options
- Create submission.csv

In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from datasets import Dataset

from transformers import (

    AutoTokenizer,

    AutoModelForSequenceClassification,

    DataCollatorWithPadding

)

from torch.utils.data import DataLoader

In [3]:
test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

sample = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

print(test.shape)

test.head()

(500, 7)


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [4]:
MODEL_PATH = "/kaggle/input/datasets/kanchandalal123/mcq-deberta-optimization-model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.cuda()

model.eval()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [5]:
rows = []

for _, row in test.iterrows():

    for option in ["A","B","C","D","E"]:

        rows.append({

            "id":row["id"],

            "option_label":option,

            "text":

            "Question: "

            + row["prompt"]

            + " [SEP] Option: "

            + row[option]

        })

ranking_test = pd.DataFrame(rows)

ranking_test.head()

,id,option_label,text
0,1,A,Question: Pick the best possible answer: What ...
1,1,B,Question: Pick the best possible answer: What ...
2,1,C,Question: Pick the best possible answer: What ...
3,1,D,Question: Pick the best possible answer: What ...
4,1,E,Question: Pick the best possible answer: What ...


In [6]:
test_ds = Dataset.from_pandas(
    ranking_test
)

In [7]:
def tokenize(batch):

    return tokenizer(

        batch["text"],

        truncation=True,

        max_length=256

    )

test_ds = test_ds.map(

    tokenize,

    batched=True
)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [8]:
test_ds.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask"

    ]
)

In [9]:
collator = DataCollatorWithPadding(
    tokenizer
)

loader = DataLoader(

    test_ds,

    batch_size=32,

    shuffle=False,

    collate_fn=collator
)

In [10]:
scores = []

with torch.no_grad():

    for batch in loader:

        batch = {

            k:v.cuda()

            for k,v in batch.items()

        }

        output = model(**batch)

        prob = F.softmax(

            output.logits,

            dim=1

        )[:,1]

        scores.extend(

            prob.cpu().numpy()

        )

In [11]:
ranking_test["score"] = scores

ranking_test.head()

,id,option_label,text,score
0,1,A,Question: Pick the best possible answer: What ...,0.999836
1,1,B,Question: Pick the best possible answer: What ...,0.000043
2,1,C,Question: Pick the best possible answer: What ...,0.000035
3,1,D,Question: Pick the best possible answer: What ...,0.000038
4,1,E,Question: Pick the best possible answer: What ...,0.000033


In [12]:
submission = (

    ranking_test

    .sort_values(

        ["id","score"],

        ascending=[True,False]

    )

    .groupby("id")["option_label"]

    .apply(

        lambda x:

        " ".join(

            x.head(3)

        )

    )

    .reset_index()
)

submission.columns = [

    "ID",

    "Prediction"

]

In [13]:
submission.head(10)

,ID,Prediction
0,1,A B D
1,2,B E C
2,3,B D E
3,4,E C D
4,5,C D B
5,6,D B A
6,7,E C D
7,8,B C E
8,9,C D A
9,10,B D E


In [14]:
submission.to_csv(

    "/kaggle/working/submission.csv",

    index=False

)

print("Submission Created Successfully")

submission.head()

Submission Created Successfully


,ID,Prediction
0,1,A B D
1,2,B E C
2,3,B D E
3,4,E C D
4,5,C D B
